# Module 6.2 — Photo Features for Your Dog

**Module 6: Vision + Deployment | Week 1 | Lab (Session 6.2)**

*Source: Refactored from archived `M05-lab4_photo_features.ipynb` (Session 29, 2026-05-18). Pairs with 6.1 — same pipeline, your dog. Backbone is ConvNeXt-Tiny (768-dim features) on multi-photo–averaged inputs; accuracy and AUC both reported. Tasks 1–4 use the M05-lab3 fill-in pattern; Task 5 is the actual application. Session 35: SHARED_6 replaces OVERLAP_5; all-dogs training.*

---

The lecture in 6.1 walked through the photo-feature pipeline on a small sample and showed how much photos add to the 6-feature SHARED set. Today you run it at scale on your own dog. By the end of this lab you'll have:

- The three-model comparison (tabular / image / combined) trained on the full PetFinder dog set with multi-photo CNN features — same numbers as the lecture, your hands on them
- Both accuracy *and* AUC reported, so you see the two metrics side by side
- A prediction for the dog you've been working with since 5.6, using the combined model — the actual application of everything you've built

No new technique here. The lecture demoed the pattern; today is reps and applying it to your dog.

In [ ]:
import os
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

## Setup — load data and bundles

Two data bundles, same as the lecture:

- **`petfinder-mini.zip`** — the M05 toolkit + dog metadata + 300-photo sample.
- **`m06-photos.zip`** — `petfinder_cnn_features.csv` (multi-photo averaged ConvNeXt features) + a `my_dog.jpg` template you'll replace with your own dog's photo in Task 5.

If you ran 6.1 in this session, the downloads are no-ops — same `fetch_bundle` helper.

In [ ]:
PETFINDER_URL  = 'CONFIGURE_ME'  # ← your instructor will give you the M05 bundle URL
M06_PHOTOS_URL = 'CONFIGURE_ME'  # ← your instructor will give you the M06 bundle URL
DATA_DIR = '/content/petfinder'

os.makedirs(DATA_DIR, exist_ok=True)

def fetch_bundle(url, marker_file, label):
    """Download + unzip a bundle if its marker file isn't already on disk."""
    if os.path.exists(f'{DATA_DIR}/{marker_file}'):
        return
    if url == 'CONFIGURE_ME':
        raise RuntimeError(f'Set {label} above to your instructor URL.')
    import gdown
    zip_path = f'{DATA_DIR}/_bundle.zip'
    gdown.download(url=url, output=zip_path, quiet=False, fuzzy=True)
    if not os.path.exists(zip_path):
        raise RuntimeError(f'{label} download failed — check URL and Drive share settings.')
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(DATA_DIR)
    os.remove(zip_path)
    if not os.path.exists(f'{DATA_DIR}/{marker_file}'):
        raise RuntimeError(f'{marker_file} not found after extraction — wrong zip?')

fetch_bundle(PETFINDER_URL,  'train.csv',                   'PETFINDER_URL')
fetch_bundle(M06_PHOTOS_URL, 'petfinder_cnn_features.csv', 'M06_PHOTOS_URL')
print(f'Bundles ready in {DATA_DIR}')


In [ ]:
df = pd.read_csv(f'{DATA_DIR}/train.csv')
dogs = df[df['Type'] == 1].copy()
dogs['adopted_in_30_days'] = (dogs['AdoptionSpeed'] <= 2).astype(int)
print(f'Dogs in dataset: {len(dogs):,}')
print(f'Adopted in 30 days (base rate): {dogs["adopted_in_30_days"].mean():.3f}')

## Pretrained CNN — the lecture's setup, in two cells

Same ConvNeXt-Tiny + identity-head trick + `weights.transforms()` preprocessing as the lecture. The full theory for *why* each piece is what it is lives in 6.1. Here we just stand the model up and define the `extract_features` helper Task 5 needs.

In [ ]:
device = torch.device(
    'cuda' if torch.cuda.is_available()
    else 'mps' if torch.backends.mps.is_available()
    else 'cpu'
)

weights = models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1
backbone = models.convnext_tiny(weights=weights)

# Drop only the final Linear; keep LayerNorm2d + Flatten so the model
# returns the 768-dim feature vector.
backbone.classifier = nn.Sequential(backbone.classifier[0], backbone.classifier[1])
backbone.eval()
backbone = backbone.to(device)
for p in backbone.parameters():
    p.requires_grad = False

preprocess = weights.transforms()

def extract_features(image_path):
    """Open one image and return its 768-dim ConvNeXt feature vector."""
    img = Image.open(image_path).convert('RGB')
    tensor = preprocess(img).unsqueeze(0).to(device)
    with torch.no_grad():
        return backbone(tensor).cpu().numpy().squeeze()

print(f'Pretrained ConvNeXt-Tiny ready on {device}.')

## Load the pre-extracted multi-photo features

The bundle's CSV holds one feature vector per dog, averaged across all the photos that listing had — same artifact the lecture used. We left-join with the dog metadata so every dog is included; the 194 dogs without photos get zero-padded CNN features. Each row ends up with both the 6 SHARED tabular features and the 768 image features.

In [ ]:
photo_df = pd.read_csv(f'{DATA_DIR}/petfinder_cnn_features.csv').set_index('PetID')
IMAGE = [f'img_{i}' for i in range(768)]
print(f'Pre-extracted multi-photo CNN features: {photo_df.shape[0]:,} dogs × {photo_df.shape[1]} features')

merged = dogs.merge(photo_df, left_on='PetID', right_index=True, how='left')
has_photo = merged[IMAGE[0]].notna().values
merged[IMAGE] = merged[IMAGE].fillna(0.0)
print(f'Dogs: {len(merged):,} total ({has_photo.sum():,} with photos, {(~has_photo).sum():,} zero-padded)')


In [ ]:
def to_shared6(d):
    out = pd.DataFrame()
    out['age_adult']      = (d['Age'] >= 12).astype(int)
    out['is_young']       = (d['Age'] < 6).astype(int)
    out['is_mixed_breed'] = ((d['Breed1'] == 307) | (d['Breed2'] != 0)).astype(int)
    out['gender_male']    = (d['Gender'] == 1).astype(int)
    out['gender_female']  = (d['Gender'] == 2).astype(int)
    out['sterilized_yes'] = (d['Sterilized'] == 1).astype(int)
    return out

SHARED_COLUMNS = ['age_adult', 'is_young', 'is_mixed_breed',
                  'gender_male', 'gender_female', 'sterilized_yes']

X_tab  = to_shared6(merged).values.astype(float)
X_img  = merged[IMAGE].values
X_both = np.concatenate([X_tab, X_img], axis=1)
y      = merged['adopted_in_30_days'].values

tr_idx, te_idx = train_test_split(
    np.arange(len(y)), test_size=0.25, random_state=42, stratify=y,
)

# Evaluate on photo-having test dogs only.
te_eval = te_idx[has_photo[te_idx]]
y_tr, y_te = y[tr_idx], y[te_eval]

def scale(X, tr, te):
    s = StandardScaler().fit(X[tr])
    return s.transform(X[tr]), s.transform(X[te]), s

X_tr_tab,  X_te_tab,  scaler_tab  = scale(X_tab,  tr_idx, te_eval)
X_tr_img,  X_te_img,  scaler_img  = scale(X_img,  tr_idx, te_eval)
X_tr_both, X_te_both, scaler_both = scale(X_both, tr_idx, te_eval)

print(f'Train: {len(tr_idx):,} (all dogs)  |  Test: {len(te_eval):,} (photo-having)')
print(f'Feature counts:  tab={X_tab.shape[1]}  img={X_img.shape[1]}  both={X_both.shape[1]}')


In [ ]:
class TabularMLP(nn.Module):
    """Same MLP shape from 5.3 / M03 — works on any feature set."""

    def __init__(self, n_features, hidden=(64, 32), dropout=0.3):
        super().__init__()
        layers = []
        prev = n_features
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


def train_mlp(X_tr, y_tr, X_te, y_te, epochs=150, seed=2):
    """Train an MLP; return (accuracy, AUC, model).

    Same shape as the lecture's `train_mlp`, but also returns the trained
    model so Task 5 can reuse it for your dog's prediction.
    """
    torch.manual_seed(seed)
    Xtr = torch.tensor(X_tr, dtype=torch.float32, device=device)
    ytr = torch.tensor(y_tr, dtype=torch.float32, device=device)
    Xte = torch.tensor(X_te, dtype=torch.float32, device=device)
    model = TabularMLP(Xtr.shape[1]).to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.BCEWithLogitsLoss()
    for _ in range(epochs):
        model.train()
        optimizer.zero_grad()
        loss_fn(model(Xtr), ytr).backward()
        optimizer.step()
    model.eval()
    with torch.no_grad():
        probs = torch.sigmoid(model(Xte)).cpu().numpy()
    preds = (probs > 0.5).astype(int)
    acc = float((preds == y_te).mean())
    auc = float(roc_auc_score(y_te, probs))
    return acc, auc, model

---

### Task 1: Tabular-only MLP

Train the MLP on the 6-feature SHARED set (`X_tr_tab`, `y_tr`, `X_te_tab`, `y_te`). Print **both** accuracy and AUC.

Suggested variable names: `acc_tab`, `auc_tab`, `model_tab`. (The parallel naming makes Tasks 2–3 easier to read.)

In [ ]:
# Task 1: your code here


---

### Task 2: Image-only MLP

Same MLP shape, just on the 768 image features (`X_tr_img`, `X_te_img`). Save `acc_img`, `auc_img`, `model_img`.

In [ ]:
# Task 2: your code here


---

### Task 3: Combined MLP

Same MLP shape, on the concatenated 6 + 768 = 774 features (`X_tr_both`, `X_te_both`). Save `acc_both`, `auc_both`, `model_both`.

**This is the model Task 5 needs** — your dog's prediction comes from it. The cell after Task 4 persists `model_both` and `scaler_both` to disk so Task 5 still works if you re-open the notebook in a new session.

In [ ]:
# Task 3: your code here


---

### Task 4: Compare

Plot the three models side by side on **both** accuracy and AUC — two subplots, same layout the lecture's bar chart used. Where do photos help most?

Hint: `plt.subplots(1, 2, figsize=(11, 4))` gives you two axes. Loop over `[(ax_acc, 'Accuracy', [acc_tab, acc_img, acc_both]), (ax_auc, 'AUC', [auc_tab, auc_img, auc_both])]` or hardcode left and right — either reads fine.

In [ ]:
# Task 4: your code here


---

### Save the combined model — given step (don't modify)

Persist Task 3's `model_both` and `scaler_both` to disk. If you didn't run Task 3 in this session, the `try` block fails and the save is silently skipped — Task 5 then loads a previously saved version (or warns you to run Task 3 first).

In [ ]:
SAVE_PATH = f'{DATA_DIR}/model_both.pth'

try:
    torch.save({
        'state_dict':       model_both.state_dict(),
        'scaler_mean':      scaler_both.mean_,
        'scaler_scale':     scaler_both.scale_,
        'shared_columns':   SHARED_COLUMNS,
        'n_image_features': 768,
    }, SAVE_PATH)
    print(f'Saved combined model + scaler to {SAVE_PATH}')
except NameError:
    print('Task 3 not yet run — save skipped. Task 5 will try to load a previous save.')


---

### Task 5: Predict for your dog

The actual application — running the combined model on the dog you've been working with since 5.6. Three steps:

1. **Upload a photo** of your dog. If you don't have one handy, the bundle ships a `my_dog.jpg` template you can use.
2. **Fill in the 6-feature SHARED row** for your dog — same schema as Task 1's training data (`age_adult`, `is_young`, `is_mixed_breed`, `gender_male`, `gender_female`, `sterilized_yes`).
3. **Predict** — extract image features, concatenate with the tabular row, scale, run through `model_both`.

The next cell is given setup: Colab-vs-local file upload guard + model load with disk fallback. The cell after that is yours — fill in your dog's SHARED row and print the predicted adoption probability.

In [ ]:
# Given setup — do not modify.
#
# Part A: get a photo file at /content/my_dog.jpg.
#   - On Colab: prompt for upload via google.colab.files.upload().
#   - Locally:  use the bundle's my_dog.jpg as a placeholder.

IN_COLAB = 'google.colab' in str(get_ipython())
MY_DOG_PATH = '/content/my_dog.jpg'

if IN_COLAB:
    from google.colab import files
    print('Choose your dog\'s photo (or cancel to use the bundled template):')
    try:
        uploaded = files.upload()
        if uploaded:
            photo_filename = next(iter(uploaded))
            with open(MY_DOG_PATH, 'wb') as f:
                f.write(uploaded[photo_filename])
            print(f'Using uploaded photo: {photo_filename}')
        else:
            raise ValueError('no upload')
    except Exception:
        # Fall back to bundle template.
        bundle_template = f'{DATA_DIR}/my_dog.jpg'
        if os.path.exists(bundle_template):
            import shutil; shutil.copy(bundle_template, MY_DOG_PATH)
            print(f'Using bundle template: {bundle_template}')
        else:
            raise RuntimeError('No upload and no bundle template available.')
else:
    # Local Jupyter: skip the upload prompt; expect the bundle template.
    bundle_template = f'{DATA_DIR}/my_dog.jpg'
    if not os.path.exists(MY_DOG_PATH) and os.path.exists(bundle_template):
        import shutil; shutil.copy(bundle_template, MY_DOG_PATH)
    print(f'Local mode — using {MY_DOG_PATH} (replace with your own dog\'s photo if you have one).')

# Part B: make sure model_both + scaler_both are available. If Task 3 was run
# this session, they're already in memory; otherwise reload from disk.
if 'model_both' not in dir() or 'scaler_both' not in dir():
    if not os.path.exists(SAVE_PATH):
        raise RuntimeError(
            'No saved model on disk and Task 3 not run this session — '
            'go back and complete Task 3 first.'
        )
    ckpt = torch.load(SAVE_PATH, map_location=device)
    n_cols = len(ckpt.get('shared_columns', ckpt.get('overlap_columns', [])))
    model_both = TabularMLP(n_cols + ckpt['n_image_features']).to(device)
    model_both.load_state_dict(ckpt['state_dict'])
    model_both.eval()
    scaler_both = StandardScaler()
    scaler_both.mean_  = ckpt['scaler_mean']
    scaler_both.scale_ = ckpt['scaler_scale']
    scaler_both.var_   = ckpt['scaler_scale'] ** 2
    scaler_both.n_features_in_ = len(scaler_both.mean_)
    print(f'Loaded combined model + scaler from {SAVE_PATH}')
else:
    print('Using in-memory model_both + scaler_both from Task 3.')

# Quick visual sanity check on the photo we'll predict on.
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(Image.open(MY_DOG_PATH).convert('RGB'))
ax.set_title('your dog'); ax.axis('off')
plt.show()


In [ ]:
# Task 5: your code here
#
# Steps (use the variables and helpers already set up above):
#   1. my_image_features = extract_features(MY_DOG_PATH)         # 768 numbers
#   2. my_shared_row     = pd.DataFrame([{...your dog's 6 features...}])
#   3. my_combined       = concatenate the two, reshape to (1, 774)
#   4. my_combined_s     = scaler_both.transform(my_combined)
#   5. logit             = model_both(torch.tensor(my_combined_s, ...))
#   6. prob              = float(torch.sigmoid(logit).cpu())
#   7. print f'Estimated probability adopted in 30 days: {prob:.1%}'


---

## Discussion

A few things worth noticing in your results on the full sample:

- **Tabular-only and image-only land in the same ballpark.** With the 6-feature SHARED set, demographics carry comparable signal to 768 CNN features — they're within about 0.01 AUC. That's a very different picture from M05's 5-feature OVERLAP, which trailed images by ~0.04. The right tabular features matter a lot.
- **Combined pulls clearly ahead.** About +0.03 AUC over either source alone. The CNN sees composition and visual quality; the tabular features encode demographics the CNN can't always reliably extract. Each fills in gaps the other can't.
- **Your dog's predicted probability** is the combined model's best guess for *that specific dog*. Treat it as a rough indicator, not a verdict. The model has plenty of uncertainty — we'll look at what the model can and can't see more carefully in M06-7's responsible-deployment lecture.

---

## What comes next

- **6.3 / 6.4** — multi-task extension. The lecture's flat concat (`[tab_6, img_768]` → one MLP) isn't the only way to combine the two feature sources. The multi-task architecture — shared CNN trunk + auxiliary head predicting `is_young` — lets `is_young` work through two channels simultaneously: tabular input AND trunk-shaping aux target. SHARED_6 + CNN + MultiTask reaches **AUC 0.727**, lifting about +0.032 over what you just measured. 6.3 builds the architecture from scratch; 6.4 is the hands-on lab.
- **6.5 / 6.6** — apply the multi-task PF head to Taiwan COA. This is the deployment deliverable M05 was building toward.
- **6.9** — capstone: the full Taiwan adoption brief for your dog with the photo-augmented model.

---

**Next session (Lecture 6.3):** Multi-task with photos. Why a shared trunk + auxiliary head outperforms flat concat, and how `is_young` working through two channels at once produces the strongest single-dataset model in M06.